In [1]:
import argparse
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import curve_fit
from scipy.sparse import diags as sp_diags
from scipy.sparse.linalg import spsolve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="spsolve requires")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CFG = dict(
    # Spectral range to retain after stitching
    wn_min=1200,
    wn_max=3000,

    # ALS baseline parameters
    als_lambda=1e5,   # smoothness (higher = smoother baseline)
    als_p=0.01,       # asymmetry (fraction of points above baseline)
    als_niter=10,

    # Peak fitting windows (center ± half_window cm-1)
    peaks=dict(
        D   =dict(center=1350, half=15),
        G   =dict(center=1580, half=12),
        D_p =dict(center=1620, half=25),   # D' shoulder
        two_D=dict(center=2700, half=55),
        DG  =dict(center=2940, half=40),   # D+G combination
    ),

    # Clustering
    n_clusters=4,

    # Spatial split: (row_start, row_end) as fraction of grid height
    # Bottom 20% of y → test, next 10% → val, rest → train
    test_rows_frac=0.20,
    val_rows_frac=0.10,

    # 1D-CNN training
    epochs=30,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    seed=42,
)


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING AND STITCHING
# ─────────────────────────────────────────────────────────────────────────────

def load_and_stitch(xlsx_path: str, cfg: dict) -> pd.DataFrame:
    """
    Load map2A and map2B sheets, stitch per pixel into a single sorted spectrum,
    crop to [wn_min, wn_max], and return a long-form DataFrame with columns:
        x, y, Raman shift, Intensity
    """
    print("Loading map2A ...")
    dfA = pd.read_excel(xlsx_path, sheet_name="map2A")
    print("Loading map2B ...")
    dfB = pd.read_excel(xlsx_path, sheet_name="map2B")

    # Overlap region: map2A extends to ~2128, map2B starts at ~2047.
    # Strategy: keep map2A for wn < 2047 (its unique range), map2B for wn >= 2047.
    # This avoids duplicating the ~80 cm-1 overlap and keeps map2B's coverage
    # of the 2D band unmodified.
    overlap_cutoff = dfB["Raman shift"].min()
    dfA_trimmed = dfA[dfA["Raman shift"] < overlap_cutoff]

    stitched = (
        pd.concat([dfA_trimmed, dfB], ignore_index=True)
        .drop_duplicates(subset=["x", "y", "Raman shift"])
        .sort_values(["x", "y", "Raman shift"])
        .reset_index(drop=True)
    )

    # Crop to requested spectral range
    stitched = stitched[
        (stitched["Raman shift"] >= cfg["wn_min"]) &
        (stitched["Raman shift"] <= cfg["wn_max"])
    ].reset_index(drop=True)

    n_pixels = stitched.groupby(["x", "y"]).ngroups
    n_pts = stitched.groupby(["x", "y"]).size().iloc[0]
    print(f"  Stitched: {n_pixels} pixels × {n_pts} spectral points "
          f"[{cfg['wn_min']}, {cfg['wn_max']}] cm⁻¹")
    return stitched


# ─────────────────────────────────────────────────────────────────────────────
# 2. BASELINE CORRECTION
# ─────────────────────────────────────────────────────────────────────────────

def als_baseline(y: np.ndarray, lam: float, p: float, niter: int) -> np.ndarray:
    """
    Asymmetric Least Squares baseline.
    Eilers & Boelens (2005): https://doi.org/10.1366/000370205774783629

    Parameters
    ----------
    y      : raw spectrum
    lam    : smoothness penalty (1e4–1e6 for Raman)
    p      : asymmetry weight (0.001–0.1; smaller → baseline hugs valleys)
    niter  : number of reweighting iterations (5–20)
    """
    L = len(y)
    e = np.ones(L)
    D = sp_diags([e, -2 * e, e], [0, 1, 2], shape=(L - 2, L), format="csc")
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(niter):
        W = sp_diags(w, 0, shape=(L, L), format="csc")
        Z = W + H
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z


def preprocess_spectra(df: pd.DataFrame, cfg: dict) -> dict:
    """
    For each (x, y) pixel:
      - Apply ALS baseline correction
      - Clip negative values to 0
      - Normalize by G-band maximum (1560–1620 cm-1)

    Returns
    -------
    dict with keys:
        positions : (N, 2) array of (x, y) coordinates
        wavenumber: (M,)  shared wavenumber axis
        spectra   : (N, M) baseline-corrected, G-normalized spectra
        raw       : (N, M) baseline-corrected but NOT normalized
    """
    pixels = df.groupby(["x", "y"], sort=False)
    wavenumber = None
    positions, spectra_corr, spectra_norm = [], [], []

    print("Preprocessing spectra (ALS baseline + G-normalization) ...")
    for (x, y), group in tqdm(pixels, total=pixels.ngroups):
        g = group.sort_values("Raman shift")
        wn = g["Raman shift"].values
        ints = g["Intensity"].values.astype(float)

        if wavenumber is None:
            wavenumber = wn

        # Baseline subtraction
        bl = als_baseline(ints, cfg["als_lambda"], cfg["als_p"], cfg["als_niter"])
        corrected = np.clip(ints - bl, 0, None)

        # G-band normalization
        g_mask = (wn >= 1560) & (wn <= 1620)
        g_max = corrected[g_mask].max()
        if g_max <= 0:
            g_max = 1.0  # avoid divide-by-zero for dead pixels
        normalized = corrected / g_max

        positions.append([x, y])
        spectra_corr.append(corrected)
        spectra_norm.append(normalized)

    return dict(
        positions=np.array(positions),
        wavenumber=wavenumber,
        raw=np.array(spectra_corr),
        spectra=np.array(spectra_norm),
    )


# ─────────────────────────────────────────────────────────────────────────────
# 3. PEAK FITTING → PHYSICS FEATURE MATRIX
# ─────────────────────────────────────────────────────────────────────────────

def lorentzian(x, A, x0, gamma):
    """Single Lorentzian: A * γ² / ((x − x₀)² + γ²)"""
    return A * gamma ** 2 / ((x - x0) ** 2 + gamma ** 2)


def fit_peak(wn: np.ndarray, spec: np.ndarray, center: float, half: float) -> dict:
    """
    Fit a single Lorentzian to a spectral window [center-half, center+half].

    Falls back to a simple window-max estimate when curve_fit fails.
    The 'fit_ok' flag in the returned dict indicates whether the Lorentzian
    converged (True) or the fallback was used (False).

    Returns dict with keys: amplitude, position, fwhm, area, snr, fit_ok
    Returns None only if the window is empty or has no positive signal.
    """
    mask = (wn >= center - half) & (wn <= center + half)
    xs, ys = wn[mask], spec[mask]
    if len(xs) < 4 or ys.max() <= 0:
        return None

    # --- Attempt Lorentzian fit ---
    try:
        A0 = ys.max()
        p0 = [A0, xs[np.argmax(ys)], 15.0]
        bounds = (
            [0,      center - half * 0.7, 1.0],
            [A0 * 5, center + half * 0.7, half],
        )
        popt, _ = curve_fit(lorentzian, xs, ys, p0=p0, bounds=bounds, maxfev=3000)
        A, x0, gamma = popt
        residuals = ys - lorentzian(xs, *popt)
        snr = A / (residuals.std() + 1e-9)
        return dict(amplitude=A, position=float(x0),
                    fwhm=float(2 * gamma), area=float(np.pi * A * gamma),
                    snr=float(snr), fit_ok=True)
    except Exception:
        pass

    # --- Fallback: window-max proxy ---
    peak_idx = np.argmax(ys)
    A = float(ys[peak_idx])
    x0 = float(xs[peak_idx])
    # Estimate half-width from the half-maximum crossing
    half_max = A / 2.0
    above = np.where(ys >= half_max)[0]
    fwhm = float(xs[above[-1]] - xs[above[0]]) if len(above) > 1 else float(half)
    return dict(amplitude=A, position=x0,
                fwhm=fwhm, area=A * fwhm,
                snr=float(A / (ys.std() + 1e-9)), fit_ok=False)


def extract_features(data: dict, cfg: dict) -> pd.DataFrame:
    """
    Fit D, G, D', 2D, D+G peaks for every pixel.
    Return a feature DataFrame with columns:
        x, y, + per-peak {amplitude, position, fwhm, area, snr}
        + derived ratios: ID_IG, I2D_IG, ID_IG_area, I2D_IG_area,
                          G_pos, twoD_pos, twoD_fwhm, IDp_IG
    """
    wn = data["wavenumber"]
    spectra = data["raw"]          # use ALS-corrected (not G-normalized) for amplitudes
    positions = data["positions"]
    peak_defs = cfg["peaks"]

    rows = []
    print("Fitting Lorentzian peaks per pixel ...")
    for i, (pos, spec) in enumerate(tqdm(zip(positions, spectra), total=len(positions))):
        row = {"x": pos[0], "y": pos[1]}
        fits = {}
        for name, pdef in peak_defs.items():
            f = fit_peak(wn, spec, pdef["center"], pdef["half"])
            fits[name] = f
            if f is not None:
                for k, v in f.items():
                    row[f"{name}_{k}"] = v
            else:
                for k in ["amplitude", "position", "fwhm", "area", "snr"]:
                    row[f"{name}_{k}"] = np.nan

        # Derived ratios (using amplitude as proxy for integrated intensity)
        G_amp   = fits["G"]["amplitude"]   if fits["G"]   else np.nan
        D_amp   = fits["D"]["amplitude"]   if fits["D"]   else np.nan
        tD_amp  = fits["two_D"]["amplitude"] if fits["two_D"] else np.nan
        Dp_amp  = fits["D_p"]["amplitude"] if fits["D_p"] else np.nan

        G_area  = fits["G"]["area"]        if fits["G"]   else np.nan
        D_area  = fits["D"]["area"]        if fits["D"]   else np.nan
        tD_area = fits["two_D"]["area"]    if fits["two_D"] else np.nan

        # Note: if G fit failed, ratios are undefined (NaN).
        # If D/2D fit failed, peak is below detection → ratio = 0.
        # This correctly encodes "no D band" as 0 rather than missing.
        G_ok = G_amp is not None and G_amp > 0

        row["ID_IG"]   = (D_amp  / G_amp if D_amp  is not None else 0.0) if G_ok else np.nan
        row["I2D_IG"]  = (tD_amp / G_amp if tD_amp is not None else 0.0) if G_ok else np.nan
        row["IDp_IG"]  = (Dp_amp / G_amp if Dp_amp is not None else 0.0) if G_ok else np.nan

        G_ar_ok = G_area is not None and G_area > 0
        row["ID_IG_area"]  = (D_area  / G_area if D_area  is not None else 0.0) if G_ar_ok else np.nan
        row["I2D_IG_area"] = (tD_area / G_area if tD_area is not None else 0.0) if G_ar_ok else np.nan

        rows.append(row)

    feat_df = pd.DataFrame(rows)
    print(f"  Feature matrix: {feat_df.shape[0]} pixels × {feat_df.shape[1]} columns")
    return feat_df


# ─────────────────────────────────────────────────────────────────────────────
# 4. PCA EXPLORATION
# ─────────────────────────────────────────────────────────────────────────────

def run_pca(data: dict, n_components: int = 6) -> dict:
    """
    PCA on the normalized spectra matrix.
    Useful for visualizing dominant spectral modes before clustering.

    Returns: dict with pca object, scores, explained_variance_ratio
    """
    print("Running PCA on normalized spectra ...")
    X = data["spectra"]
    pca = PCA(n_components=n_components, random_state=42)
    scores = pca.fit_transform(X)
    print(f"  Top {n_components} PCs explain "
          f"{pca.explained_variance_ratio_.sum()*100:.1f}% of variance")
    return dict(pca=pca, scores=scores, evr=pca.explained_variance_ratio_)


# ─────────────────────────────────────────────────────────────────────────────
# 5. UNSUPERVISED K-MEANS CLUSTERING
# ─────────────────────────────────────────────────────────────────────────────

def cluster_pixels(feat_df: pd.DataFrame, pca_result: dict, cfg: dict) -> np.ndarray:
    """
    K-means clustering on a combination of:
      - PCA scores (captures full spectral shape)
      - Physics ratios (ID/IG, I2D/IG) for interpretability

    Labels are reordered so cluster 0 = lowest ID/IG (most pristine).
    Returns: labels array of shape (N,)
    """
    ratio_cols = ["ID_IG", "I2D_IG", "IDp_IG"]
    ratio_data = feat_df[ratio_cols].fillna(feat_df[ratio_cols].median())

    # Combine PCA scores (first 4 PCs) + scaled ratios
    pca_scores = pca_result["scores"][:, :4]
    scaler = StandardScaler()
    ratio_scaled = scaler.fit_transform(ratio_data)
    X_cluster = np.hstack([pca_scores, ratio_scaled])

    print(f"K-means clustering (k={cfg['n_clusters']}) ...")
    km = KMeans(n_clusters=cfg["n_clusters"], random_state=42, n_init=20)
    labels = km.fit_predict(X_cluster)

    # Reorder clusters by ascending median ID/IG (0 = cleanest)
    cluster_idg = {c: feat_df["ID_IG"].values[labels == c].mean() for c in range(cfg["n_clusters"])}
    order = sorted(cluster_idg, key=cluster_idg.get)
    remap = {old: new for new, old in enumerate(order)}
    labels = np.array([remap[l] for l in labels])

    for c in range(cfg["n_clusters"]):
        mask = labels == c
        print(f"  Cluster {c}: {mask.sum():4d} pixels | "
              f"ID/IG={feat_df['ID_IG'].values[mask].mean():.3f} | "
              f"I2D/IG={feat_df['I2D_IG'].values[mask].mean():.3f}")
    return labels


# ─────────────────────────────────────────────────────────────────────────────
# 6. SPATIAL TRAIN / VAL / TEST SPLIT
# ─────────────────────────────────────────────────────────────────────────────

def spatial_split(positions: np.ndarray, cfg: dict) -> dict:
    """
    Split pixels by contiguous y-axis bands to avoid data leakage.

      test  → bottom (cfg['test_rows_frac'])  of y range
      val   → next   (cfg['val_rows_frac'])   above test
      train → everything above val

    This ensures test/val pixels were never neighbors of training pixels
    during feature extraction, preventing spatial autocorrelation leakage.

    Returns: dict with 'train', 'val', 'test' index arrays
    """
    ys = positions[:, 1]
    y_min, y_max = ys.min(), ys.max()
    y_range = y_max - y_min

    y_test_thresh = y_min + cfg["test_rows_frac"] * y_range
    y_val_thresh  = y_test_thresh + cfg["val_rows_frac"] * y_range

    test_idx  = np.where(ys <= y_test_thresh)[0]
    val_idx   = np.where((ys > y_test_thresh) & (ys <= y_val_thresh))[0]
    train_idx = np.where(ys > y_val_thresh)[0]

    n = len(positions)
    print(f"Spatial split → train: {len(train_idx)} ({len(train_idx)/n*100:.0f}%) | "
          f"val: {len(val_idx)} ({len(val_idx)/n*100:.0f}%) | "
          f"test: {len(test_idx)} ({len(test_idx)/n*100:.0f}%)")
    return dict(train=train_idx, val=val_idx, test=test_idx)


# ─────────────────────────────────────────────────────────────────────────────
# 7. DATASET + DATALOADER
# ─────────────────────────────────────────────────────────────────────────────

class RamanDataset(Dataset):
    """PyTorch Dataset for 1D Raman spectra."""

    def __init__(self, spectra: np.ndarray, labels: np.ndarray,
                 augment: bool = False):
        self.spectra = torch.tensor(spectra, dtype=torch.float32).unsqueeze(1)
        self.labels  = torch.tensor(labels,  dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.spectra[idx]
        y = self.labels[idx]
        if self.augment:
            x = self._augment(x)
        return x, y

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        """
        Physics-informed augmentation:
          - Gaussian noise (simulate photon shot noise)
          - Intensity scaling (simulate laser power variation)
          - Spectral shift (simulate wavenumber calibration drift)
        """
        # Gaussian noise
        x = x + torch.randn_like(x) * 0.02

        # Intensity scaling ±15%
        scale = 0.85 + torch.rand(1) * 0.30
        x = x * scale

        # Spectral shift ±3 pixels (circular roll)
        shift = int((torch.rand(1) * 6 - 3).item())
        x = torch.roll(x, shift, dims=-1)

        return x


def build_loaders(data: dict, labels: np.ndarray, splits: dict, cfg: dict):
    """
    Build train/val/test DataLoaders with class-balanced sampling for training.
    """
    train_idx, val_idx, test_idx = splits["train"], splits["val"], splits["test"]
    X = data["spectra"]

    train_ds = RamanDataset(X[train_idx], labels[train_idx], augment=True)
    val_ds   = RamanDataset(X[val_idx],   labels[val_idx],   augment=False)
    test_ds  = RamanDataset(X[test_idx],  labels[test_idx],  augment=False)

    # Weighted sampler to handle class imbalance
    train_labels = labels[train_idx]
    class_counts = np.bincount(train_labels)
    weights = 1.0 / class_counts[train_labels]
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], sampler=sampler)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"], shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=cfg["batch_size"], shuffle=False)
    return train_loader, val_loader, test_loader


# ─────────────────────────────────────────────────────────────────────────────
# 8. 1D-CNN MODEL
# ─────────────────────────────────────────────────────────────────────────────

class RamanCNN1D(nn.Module):
    """
    1D Convolutional Neural Network for Raman spectrum classification.

    Architecture:
        3 × (Conv1D → BatchNorm → ReLU → MaxPool)
        → GlobalAvgPool
        → FC(256) → Dropout → FC(n_classes)

    Receptive field:
        Layer 1: kernel=15 → captures ~15 cm-1 features (narrow peaks)
        Layer 2: kernel=9  → captures ~72 cm-1 features (peak groups)
        Layer 3: kernel=7  → captures ~288 cm-1 features (band regions)

    The gradually increasing receptive field lets the network learn
    features at multiple spectral scales — from individual peak widths
    to inter-band ratios.
    """

    def __init__(self, n_channels: int, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(1,  32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 2
            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 3
            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 4
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pooling
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

    def get_feature_maps(self, x):
        """Return feature maps before global pooling (for Grad-CAM)."""
        return self.features(x)


# ─────────────────────────────────────────────────────────────────────────────
# 9. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train_model(model: nn.Module, train_loader, val_loader,
                cfg: dict, device: torch.device) -> dict:
    """
    Training loop with:
      - Adam optimizer + cosine LR annealing
      - Early stopping (patience = 8 epochs)
      - Saves best checkpoint based on val accuracy
    """
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"]
    )
    criterion = nn.CrossEntropyLoss()

    history = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[])
    best_val_acc, patience_count, best_state = 0.0, 0, None
    patience = 8

    for epoch in range(cfg["epochs"]):
        # --- Training ---
        model.train()
        total_loss, total_correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
            total_correct += (logits.argmax(1) == y_batch).sum().item()
            total += len(y_batch)
        train_loss = total_loss / total
        train_acc  = total_correct / total

        # --- Validation ---
        model.eval()
        vloss, vcorrect, vtotal = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                vloss += loss.item() * len(y_batch)
                vcorrect += (logits.argmax(1) == y_batch).sum().item()
                vtotal += len(y_batch)
        val_loss = vloss / vtotal
        val_acc  = vcorrect / vtotal

        scheduler.step()
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"  Epoch {epoch+1:3d}/{cfg['epochs']} | "
              f"loss {train_loss:.4f} → {val_loss:.4f} | "
              f"acc {train_acc*100:.1f}% → {val_acc*100:.1f}% ")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    print(f"Best val acc: {best_val_acc*100:.2f}%")
    return history


# ─────────────────────────────────────────────────────────────────────────────
# 10. GRAD-CAM (1D)
# ─────────────────────────────────────────────────────────────────────────────

class GradCAM1D:
    """
    Grad-CAM for 1D CNNs.

    Highlights wavenumber regions the network uses to distinguish classes.
    Compare the highlighted regions to known band positions to validate
    that the network has learned physically meaningful features.

    Reference: Selvaraju et al. (2017), https://arxiv.org/abs/1610.02391
    """

    def __init__(self, model: nn.Module):
        self.model = model
        self.gradients = None
        self.activations = None
        model.features[-3].register_forward_hook(self._save_activation)
        model.features[-3].register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def compute(self, x: torch.Tensor, class_idx: int = None) -> np.ndarray:
        """
        Parameters
        ----------
        x         : (1, 1, L) input spectrum tensor
        class_idx : target class (None → argmax of prediction)

        Returns
        -------
        cam : (L,) normalized CAM heatmap, 0–1
        """
        self.model.eval()
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        score = logits[0, class_idx]
        self.model.zero_grad()
        score.backward()

        # Global average pool of gradients over spectral dimension
        weights = self.gradients.mean(dim=-1, keepdim=True)    # (1, C, 1)
        cam_raw = (weights * self.activations).sum(dim=1)       # (1, L')
        cam_raw = F.relu(cam_raw).squeeze().cpu().numpy()

        # Upsample to input length
        cam_up = np.interp(
            np.linspace(0, len(cam_raw) - 1, x.shape[-1]),
            np.arange(len(cam_raw)),
            cam_raw,
        )
        # Normalize
        cam_min, cam_max = cam_up.min(), cam_up.max()
        if cam_max > cam_min:
            cam_up = (cam_up - cam_min) / (cam_max - cam_min)
        return cam_up


# ─────────────────────────────────────────────────────────────────────────────
# 11. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def evaluate(model: nn.Module, loader, device: torch.device) -> tuple:
    """Return (all_preds, all_true) arrays for the given DataLoader."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(y_batch.numpy())
    return np.array(preds), np.array(trues)


# ─────────────────────────────────────────────────────────────────────────────
# 12. VISUALIZATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def make_grid_map(positions: np.ndarray, values: np.ndarray):
    """Convert (N, 2) positions + (N,) values to 2D grid for imshow."""
    xs = np.unique(positions[:, 0])
    ys = np.unique(positions[:, 1])
    xi = {v: i for i, v in enumerate(xs)}
    yi = {v: i for i, v in enumerate(ys)}
    grid = np.full((len(ys), len(xs)), np.nan)
    for pos, val in zip(positions, values):
        grid[yi[pos[1]], xi[pos[0]]] = val
    return grid, xs, ys


def save_spatial_maps(data: dict, feat_df: pd.DataFrame, labels: np.ndarray,
                      splits: dict, out_dir: Path):
    """Save publication-style spatial maps."""
    pos = data["positions"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    maps = [
        ("I₂D / I_G ratio",  feat_df["I2D_IG"].values,  "plasma"),
        ("I_D / I_G ratio",   feat_df["ID_IG"].values,   "viridis"),
        ("G-band position\n(cm⁻¹)", feat_df["G_position"].values, "RdYlBu_r"),
        ("2D-band FWHM\n(cm⁻¹)",    feat_df["two_D_fwhm"].values, "hot"),
        ("Cluster label\n(0=pristine)", labels.astype(float), "tab10"),
        ("Train/Val/Test\n(0/1/2)", None, "coolwarm"),
    ]

    # Build train/val/test map
    split_labels = np.zeros(len(pos))
    split_labels[splits["val"]]  = 1
    split_labels[splits["test"]] = 2
    maps[-1] = ("Train/Val/Test\n(0=train, 1=val, 2=test)", split_labels, "coolwarm")

    for ax, (title, values, cmap) in zip(axes.flat, maps):
        grid, xs, ys = make_grid_map(pos, values)
        im = ax.imshow(grid, origin="lower", aspect="auto", cmap=cmap,
                       extent=[xs.min(), xs.max(), ys.min(), ys.max()])
        ax.set_xlabel("x (µm)", fontsize=10)
        ax.set_ylabel("y (µm)", fontsize=10)
        ax.set_title(title, fontsize=11)
        plt.colorbar(im, ax=ax, fraction=0.04)

    plt.suptitle("Raman Map Analysis — Graphene/Graphite", fontsize=13, y=1.01)
    plt.tight_layout()
    path = out_dir / "spatial_maps.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def save_training_curves(history: dict, out_dir: Path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(history["train_loss"], label="train")
    ax1.plot(history["val_loss"],   label="val")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()
    ax1.set_title("Loss")
    ax2.plot(np.array(history["train_acc"]) * 100, label="train")
    ax2.plot(np.array(history["val_acc"])   * 100, label="val")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.legend()
    ax2.set_title("Accuracy")
    plt.tight_layout()
    path = out_dir / "training_curves.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def save_gradcam_figure(model: nn.Module, data: dict, labels: np.ndarray,
                        splits: dict, out_dir: Path, device: torch.device,
                        n_per_class: int = 3):
    """
    For each cluster class, plot mean spectrum + Grad-CAM heatmap overlay.
    Shows which wavenumber regions drive the classification decision.
    """
    wn = data["wavenumber"]
    X  = torch.tensor(data["spectra"], dtype=torch.float32).unsqueeze(1)
    cam = GradCAM1D(model)
    n_classes = len(np.unique(labels))
    band_markers = {"D": 1350, "G": 1580, "D'": 1620, "2D": 2700}

    fig, axes = plt.subplots(n_classes, 1, figsize=(12, 4 * n_classes), sharex=True)
    if n_classes == 1:
        axes = [axes]

    for c, ax in enumerate(axes):
        idx_c = np.where(labels == c)[0]
        # Mean spectrum for this class
        mean_spec = data["spectra"][idx_c].mean(axis=0)
        # Grad-CAM averaged over a few samples
        cams = []
        for i in idx_c[:n_per_class]:
            xi = X[i:i+1].to(device)
            cams.append(cam.compute(xi, class_idx=c))
        mean_cam = np.array(cams).mean(axis=0) if cams else np.zeros(len(wn))

        ax.plot(wn, mean_spec, color="#7F77DD", linewidth=1.5, label=f"Cluster {c} mean")
        ax2 = ax.twinx()
        ax2.fill_between(wn, mean_cam, alpha=0.35, color="#D85A30", label="Grad-CAM")
        ax2.set_ylim(0, 1.5)
        ax2.set_ylabel("CAM weight", fontsize=9, color="#D85A30")
        ax2.tick_params(axis="y", colors="#D85A30")

        for band, pos in band_markers.items():
            if wn.min() < pos < wn.max():
                ax.axvline(pos, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
                ax.text(pos + 5, ax.get_ylim()[1] * 0.9, band, fontsize=8, color="gray")

        ax.set_ylabel("Norm. intensity", fontsize=10)
        ax.set_title(f"Cluster {c} (n={len(idx_c)})", fontsize=11)
        ax.legend(loc="upper left", fontsize=9)

    axes[-1].set_xlabel("Raman shift (cm⁻¹)", fontsize=11)
    plt.suptitle("1D-CNN Grad-CAM — learned spectral features per class", fontsize=12)
    plt.tight_layout()
    path = out_dir / "gradcam.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Raman Map ML Pipeline")
    parser.add_argument("--data", type=str, default="Raman_maps.xlsx",
                        help="Path to the Raman_maps.xlsx file")
    parser.add_argument("--out",  type=str, default="results",
                        help="Output directory for all saved files")
    parser.add_argument("--no-train", action="store_true",
                        help="Skip CNN training (run preprocessing + clustering only)")
    args, unknown = parser.parse_known_args()

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(CFG["seed"])
    np.random.seed(CFG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    # ── Step 1: Load + stitch ────────────────────────────────────────────────
    df = load_and_stitch(args.data, CFG)

    # ── Step 2: Preprocess ───────────────────────────────────────────────────
    data = preprocess_spectra(df, CFG)

    # ── Step 3: Physics features ─────────────────────────────────────────────
    feat_df = extract_features(data, CFG)
    feat_df.to_csv(out_dir / "features.csv", index=False)
    print(f"  Saved: {out_dir}/features.csv\n")

    # ── Step 4: PCA ──────────────────────────────────────────────────────────
    pca_result = run_pca(data)

    # ── Step 5: Clustering → pseudo-labels ───────────────────────────────────
    labels = cluster_pixels(feat_df, pca_result, CFG)
    feat_df["cluster"] = labels
    print()

    # ── Step 6: Spatial split ────────────────────────────────────────────────
    splits = spatial_split(data["positions"], CFG)
    np.savez(out_dir / "splits.npz", **splits)
    print(f"  Saved: {out_dir}/splits.npz\n")

    # ── Step 7: Spatial maps ─────────────────────────────────────────────────
    print("Saving spatial maps ...")
    save_spatial_maps(data, feat_df, labels, splits, out_dir)
    print()

    if args.no_train:
        print("Skipping CNN training (--no-train). Done.")
        return

    # ── Step 8: Build dataloaders ────────────────────────────────────────────
    train_loader, val_loader, test_loader = build_loaders(data, labels, splits, CFG)

    # ── Step 9: Train 1D-CNN ─────────────────────────────────────────────────
    n_channels = data["spectra"].shape[1]
    n_classes  = CFG["n_clusters"]
    model = RamanCNN1D(n_channels, n_classes).to(device)
    print(f"\nModel: {sum(p.numel() for p in model.parameters()):,} parameters")
    print("Training 1D-CNN ...")
    history = train_model(model, train_loader, val_loader, CFG, device)

    # ── Step 10: Save model + curves ─────────────────────────────────────────
    ckpt_path = out_dir / "model.pt"
    torch.save(
        {
            "model_state": model.state_dict(),
            "cfg": CFG,
            "n_channels": n_channels,
            "n_classes": n_classes,
            "history": history,
        },
        ckpt_path,
    )
    print(f"\n  Saved: {ckpt_path}")
    save_training_curves(history, out_dir)

    # ── Step 11: Test set evaluation ─────────────────────────────────────────
    preds, trues = evaluate(model, test_loader, device)
    print("\nTest set classification report:")
    print(
        classification_report(
            trues, preds, target_names=[f"Cluster {c}" for c in range(n_classes)]
        )
    )

    # ── Step 12: Grad-CAM ────────────────────────────────────────────────────
    print("Computing Grad-CAM ...")
    save_gradcam_figure(model, data, labels, splits, out_dir, device)

    print(f"\nAll outputs saved to: {out_dir}/")
    print("Files:")
    for f in sorted(out_dir.iterdir()):
        print(f"  {f.name}")


if __name__ == "__main__":
    main()

Device: cpu

Loading map2A ...
Loading map2B ...
  Stitched: 945 pixels × 1799 spectral points [1200, 3000] cm⁻¹
Preprocessing spectra (ALS baseline + G-normalization) ...


100%|██████████| 945/945 [00:37<00:00, 25.22it/s]


Fitting Lorentzian peaks per pixel ...


100%|██████████| 945/945 [00:33<00:00, 27.98it/s]


  Feature matrix: 945 pixels × 37 columns
  Saved: results/features.csv

Running PCA on normalized spectra ...
  Top 6 PCs explain 65.9% of variance
K-means clustering (k=4) ...


C:\Users\prith\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


  Cluster 0:  413 pixels | ID/IG=0.102 | I2D/IG=0.277
  Cluster 1:  194 pixels | ID/IG=0.574 | I2D/IG=0.555
  Cluster 2:  293 pixels | ID/IG=0.642 | I2D/IG=0.822
  Cluster 3:   45 pixels | ID/IG=0.693 | I2D/IG=1.397

Spatial split → train: 648 (69%) | val: 108 (11%) | test: 189 (20%)
  Saved: results/splits.npz

Saving spatial maps ...
  Saved: results\spatial_maps.png


Model: 274,948 parameters
Training 1D-CNN ...
  Epoch   1/30 | loss 0.8519 → 1.1800 | acc 69.9% → 36.1% 
  Epoch   2/30 | loss 0.5298 → 1.7600 | acc 79.2% → 14.8% 
  Epoch   3/30 | loss 0.4157 → 0.7307 | acc 86.7% → 68.5% 
  Epoch   4/30 | loss 0.3181 → 0.3412 | acc 88.9% → 87.0% 
  Epoch   5/30 | loss 0.2412 → 0.1978 | acc 92.1% → 90.7% 
  Epoch   6/30 | loss 0.2256 → 0.2106 | acc 92.6% → 89.8% 
  Epoch   7/30 | loss 0.1833 → 0.1789 | acc 95.7% → 93.5% 
  Epoch   8/30 | loss 0.1416 → 0.1409 | acc 95.8% → 96.3% 
  Epoch   9/30 | loss 0.1417 → 0.2575 | acc 94.8% → 91.7% 
  Epoch  10/30 | loss 0.1971 → 0.2382 | acc 94.1%